In [ ]:
from ipyprogressivis.widgets.scatterplot import Scatterplot
sc = Scatterplot(enable_centroids=True)
import warnings
warnings.filterwarnings('ignore')

In [ ]:
from progressivis.cluster import MBKMeans, MBKMeansFilter
from progressivis.io.api import SimpleCSVLoader
from progressivis.vis import MCScatterPlot
from progressivis.datasets import get_dataset
from progressivis import Sink
import pandas as pd
import numpy as np
import os.path
import asyncio as aio
from progressivis.datasets import get_dataset
X_COL = "pickup_longitude"
Y_COL = "pickup_latitude"
data = SimpleCSVLoader("https://www.aviz.fr/nyc-taxi/yellow_tripdata_2015-01_clean.csv.bz2", usecols=[X_COL, Y_COL])
n_clusters = 5
mbkmeans = MBKMeans(n_clusters=n_clusters, 
                    batch_size=100, tol=0.01, is_input=False)
mbkmeans.create_dependent_modules(data)
classes = []
for i in range(n_clusters):
    cname = f"k{i}"
    filt = MBKMeansFilter(i)
    filt.create_dependent_modules(mbkmeans, data, 'result')
    sink = Sink()
    sink.input.inp = filt.output.result
    classes.append({'name': cname, 'x_column': X_COL,
                    'y_column': Y_COL, 'sample': mbkmeans if i==0 else None,
                    'sample_slot': 'result',
                    'input_module': filt, 'input_slot': 'result'})
sp = MCScatterPlot(classes=classes, queryable=False)
sp.create_dependent_modules()
sp.move_point = mbkmeans.dep.moved_center # for input a bt management
show = sc.link_module(sp)
sp.scheduler.task_start();

In [ ]:
display(sc)